# Proyecto final herramientas de programacion 2

In [1]:
# ==========================================
# FASE 1: LIBRERÍAS Y CLASES DE LOS BARCOS
# ==========================================

import random # Librería necesaria para probabilidades y eventos

class Barco:
    def __init__(self, nombre, tipo):
        self.nombre = nombre
        self.tipo = tipo.lower() # 'portaaviones', 'inteligencia', 'ingeniero', 'pesquero'
        self.nivel = 1
        self.vivo = True

        # --- ESTADÍSTICAS INICIALES (NIVEL 1) ---
        if self.tipo == 'portaaviones':
            self.vida_max = 100
            self.vida_actual = 100
            self.danio = 30
        elif self.tipo == 'inteligencia':
            self.vida_max = 40
            self.vida_actual = 40
            self.danio = 20
            self.bono_precision = 0.30  # Mejora 30% la precisión
        elif self.tipo == 'ingeniero':
            self.vida_max = 80
            self.vida_actual = 80
            self.reparacion = 25
            self.submarino_desbloqueado = False
            self.ataque_torpedo = False
        elif self.tipo == 'pesquero':
            self.vida_max = 60
            self.vida_actual = 60
            self.creditos_generados = 500
            self.abastecimiento_eficiente = False

    def subir_nivel(self):
        """Mejora las estadísticas del barco según las reglas del juego."""
        if self.nivel >= 3:
            print(f"[{self.nombre}] ya está en su nivel máximo (Nivel 3).")
            return

        self.nivel += 1
        print(f"🔼 ¡[{self.nombre}] ha subido al Nivel {self.nivel}!")

        # --- MEJORAS POR TIPO Y NIVEL ---
        if self.tipo == 'portaaviones':
            if self.nivel == 2:
                self.vida_max = 125
                self.danio = 35
            elif self.nivel == 3:
                self.vida_max = 150
                self.danio = 45
            self.vida_actual = self.vida_max # Se cura al tope al subir de nivel

        elif self.tipo == 'inteligencia':
            if self.nivel == 2:
                self.danio = 25
                self.bono_precision = 0.35
            elif self.nivel == 3:
                self.vida_max = 50
                self.bono_precision = 0.50
            self.vida_actual = self.vida_max

        elif self.tipo == 'ingeniero':
            if self.nivel == 2:
                self.reparacion = 35
                self.submarino_desbloqueado = True
                print(f"   > Se desbloqueó 'Submarino' (1 de vida, 100% de precisión aliada).")
            elif self.nivel == 3:
                self.reparacion = 50
                self.submarino_desbloqueado = False
                self.ataque_torpedo = True
                print(f"   > Pierde 'Submarino', pero desbloquea 'Ataque Torpedo' (60 de daño).")

        elif self.tipo == 'pesquero':
            if self.nivel == 2:
                self.vida_max = 75
                self.creditos_generados = 650
            elif self.nivel == 3:
                self.creditos_generados = 750
                self.abastecimiento_eficiente = True
                print(f"   > Desbloquea 'Abastecimiento Eficiente' (+15% de munición).")
            self.vida_actual = self.vida_max

    def recibir_danio(self, cantidad):
        """Resta vida al barco y verifica si fue destruido."""
        if not self.vivo:
            return

        self.vida_actual -= cantidad
        if self.vida_actual <= 0:
            self.vida_actual = 0
            self.vivo = False
            print(f"💥 ¡ALERTA! El barco [{self.nombre}] ha sido DESTRUIDO.")
        else:
            print(f"⚠️ [{self.nombre}] recibió {cantidad} de daño. Vida restante: {self.vida_actual}/{self.vida_max}")

    def recibir_reparacion(self, cantidad):
        """Suma vida al barco sin sobrepasar el máximo."""
        if not self.vivo:
            print(f"[{self.nombre}] está destruido, no se puede reparar.")
            return

        self.vida_actual += cantidad
        if self.vida_actual > self.vida_max:
            self.vida_actual = self.vida_max
        print(f"🔧 [{self.nombre}] ha sido reparado. Vida actual: {self.vida_actual}/{self.vida_max}")

In [2]:
# ========================================== # Encabezado visual para organizar la celda
# FASE 2: CLASE JUGADOR Y GESTIÓN DE TIENDA  # Título de la sección
# ========================================== # Cierre del encabezado visual

class Jugador: # Definimos la clase Jugador que servirá tanto para ti como para la máquina
    def __init__(self, nombre, es_bot=False): # Método constructor que inicializa los atributos al crear un jugador
        self.nombre = nombre # Asignamos el nombre del jugador (ej. "Jugador 1" o "IA")
        self.es_bot = es_bot # Booleano para identificar si el jugador tomará decisiones automáticas
        self.creditos = 750 # Asignamos los 750 créditos iniciales exigidos en las reglas

        # Diccionario para gestionar el inventario de munición, todas inician en 0
        self.municion = {"torpedo": 0, "cañon": 0, "artilleria": 0, "bombardeo": 0}

        # Diccionario constante con los costos oficiales de la armería
        self.precios_tienda = {"torpedo": 1500, "cañon": 500, "artilleria": 750, "bombardeo": 900}

        # Creamos la flota instanciando 4 objetos de la clase Barco (Fase 1) y los guardamos en una lista
        self.flota = [
            Barco(f"Portaaviones", "portaaviones"), # Instancia del Portaaviones
            Barco(f"Inteligencia", "inteligencia"), # Instancia del Barco de Inteligencia
            Barco(f"Ingeniero", "ingeniero"), # Instancia del Barco de Ingenieros
            Barco(f"Pesquero", "pesquero") # Instancia del Barco Pesquero
        ]

    def barcos_vivos(self): # Método para obtener únicamente los barcos que no han sido destruidos
        # APLICACIÓN DE LIST COMPREHENSION: Filtra la lista 'flota' devolviendo solo los que tienen self.vivo en True
        return [barco for barco in self.flota if barco.vivo]

    def mostrar_estado(self): # Método para imprimir la interfaz de estado en la consola
        print(f"\n--- ESTADO DE FLOTA Y RECURSOS: {self.nombre.upper()} ---") # Imprime título con el nombre en mayúsculas
        print(f"💰 Créditos disponibles: {self.creditos}") # Muestra los créditos actuales

        # LIST COMPREHENSION 2: Crea una lista de textos con formato 'Arma: Cantidad' iterando el diccionario de munición
        municion_texto = [f"{arma.capitalize()}: {cant}" for arma, cant in self.municion.items()]
        print(f"📦 Munición: {' | '.join(municion_texto)}") # Une la lista de textos con barras separadoras e imprime

        print("🚢 Estado de los barcos:") # Encabezado para la lista de la flota
        for barco in self.flota: # Ciclo for para recorrer cada uno de los 4 barcos
            # OPERADOR TERNARIO: Si el barco está vivo guarda su vida, si no, guarda el texto 'DESTRUIDO'
            estado = f"{barco.vida_actual}/{barco.vida_max} HP" if barco.vivo else "DESTRUIDO 💥"
            print(f"   > {barco.nombre} (Nv {barco.nivel}) -> {estado}") # Imprime los datos del barco iterado

    def comprar_municion(self, tipo_arma): # Método para procesar transacciones en la tienda
        tipo_arma = tipo_arma.lower() # Aseguramos que el texto esté en minúsculas para buscar en el diccionario

        if tipo_arma not in self.precios_tienda: # Verifica si el jugador escribió un arma que no existe
            print("❌ Arma no válida en el catálogo.") # Notifica el error de escritura
            return False # Retorna False para indicar que la acción no consumió turno

        costo = self.precios_tienda[tipo_arma] # Extrae el valor en créditos del arma seleccionada

        if self.creditos >= costo: # Condicional: ¿El jugador tiene dinero suficiente?
            self.creditos -= costo # Resta el dinero del saldo del jugador

            # LIST COMPREHENSION 3: Busca específicamente el barco pesquero dentro de la flota
            pesquero = [b for b in self.flota if b.tipo == 'pesquero'][0]

            # LÓGICA DE HABILIDAD: Si el pesquero está vivo y es nivel 3, el multiplicador es 1.15 (15% más), sino es 1.0
            multiplicador = 1.15 if (pesquero.vivo and pesquero.abastecimiento_eficiente) else 1.0

            # Suma la munición al inventario multiplicada por la bonificación (redondeada a 2 decimales por si acaso)
            self.municion[tipo_arma] += round(1 * multiplicador, 2)

            # Notifica que la transacción fue un éxito y muestra el vuelto
            print(f"✅ ¡Compra exitosa! [{self.nombre}] adquirió {tipo_arma}. Saldo restante: {self.creditos}")
            return True # Retorna True, lo que significará más adelante que la acción fue válida y gasta el turno

        else: # Entra aquí si el jugador no tiene el dinero suficiente
            # Imprime mensaje de rechazo mostrando el déficit
            print(f"⛔ Fondos insuficientes. {tipo_arma.capitalize()} cuesta {costo} y tienes {self.creditos}.")
            return False # Retorna False para que el jugador intente otra acción

# ========================================== # Separador visual final
# INICIALIZACIÓN DE LOS DOS JUGADORES        # Título descriptivo
# ========================================== # Separador visual final

# Instanciamos al jugador humano llamando a la clase Jugador
jugador_humano = Jugador("Comandante Humano", es_bot=False)

# Instanciamos a la máquina activando el parámetro es_bot en True
jugador_maquina = Jugador("Inteligencia Artificial", es_bot=True)

In [3]:
# ========================================== # Encabezado visual para organizar la celda
# FASE 3: MOTOR DE EVENTOS CLIMÁTICOS        # Título de la sección
# ========================================== # Cierre del encabezado visual

import random # Importamos la librería random (por si no se cargó en celdas anteriores)

class GestorClima: # Definimos la clase que controlará las condiciones de cada turno
    def __init__(self): # Método constructor que inicializa los valores por defecto
        self.evento_actual = "Día Normal" # Guardamos el nombre del evento activo
        self.acciones_permitidas = 2 # El estándar de acciones por turno es 2
        self.penalizacion_portaaviones = 0.0 # Porcentaje de penalización de precisión, inicia en 0
        self.ataques_bloqueados = False # Booleano que define si se permite atacar o no en este turno

    def resetear_condiciones(self): # Método para limpiar los efectos del turno anterior
        self.acciones_permitidas = 2 # Volvemos a las 2 acciones predeterminadas
        self.penalizacion_portaaviones = 0.0 # Quitamos cualquier penalización de precisión
        self.ataques_bloqueados = False # Desbloqueamos los ataques
        self.evento_actual = "Día Normal" # Reiniciamos el texto del evento

    def sortear_clima(self): # Método que se ejecutará al inicio de cada ronda
        self.resetear_condiciones() # Llamamos a la función de limpieza antes de aplicar un nuevo clima

        # LISTA 1: Nombres de los eventos posibles, incluyendo el 10% restante como 'Normal'
        tipos_eventos = ["Tormenta", "Mar en calma", "Accidente en cubierta", "Cese al fuego temporal", "Normal"]

        # LISTA 2: Pesos porcentuales exactos que definiste en las reglas (suman 100)
        probabilidades = [20, 15, 35, 20, 10]

        # USO DE RANDOM.CHOICES: Selecciona un evento basado en los pesos porcentuales y devuelve una lista de 1 elemento
        evento_seleccionado = random.choices(tipos_eventos, weights=probabilidades, k=1)[0]

        self.evento_actual = evento_seleccionado # Guardamos el evento sorteado en la variable de la clase

        # CONDICIONALES MULTIPLES: Evaluamos qué evento salió para aplicar sus reglas específicas
        if evento_seleccionado == "Tormenta": # Revisa si cayó el evento Tormenta
            self.acciones_permitidas = 1 # Sobrescribe las acciones del turno dejándolas en solo 1
            print("🌩️ ¡ALERTA DE TORMENTA! Las condiciones son extremas. Solo tienes 1 acción este turno.") # Notifica al jugador

        elif evento_seleccionado == "Mar en calma": # Revisa si cayó el evento Mar en calma
            self.acciones_permitidas = 3 # Aumenta las acciones del jugador a 3
            print("🌊 ¡MAR EN CALMA! La tripulación es más eficiente. Tienes 3 acciones este turno.") # Notifica la ventaja

        elif evento_seleccionado == "Accidente en cubierta": # Revisa si cayó el evento Accidente
            self.penalizacion_portaaviones = 0.05 # Guarda un 5% (0.05) de penalización para restar en la fórmula de ataque
            print("🔥 ¡ACCIDENTE EN CUBIERTA! Los Portaaviones pierden 5% de precisión en este turno.") # Notifica el peligro

        elif evento_seleccionado == "Cese al fuego temporal": # Revisa si cayó Cese al fuego
            self.ataques_bloqueados = True # Cambia el booleano a True, lo que impedirá usar el comando de disparo
            print("🕊️ ¡CESE AL FUEGO! Órdenes del alto mando: Está prohibido atacar durante este turno. Solo apoyo logístico.") # Notifica la restricción

        else: # Si no cayó ninguno de los anteriores, es porque cayó 'Normal'
            print("☀️ El clima está despejado. Condiciones normales de combate.") # Notifica que no hay alteraciones

        # Retorna el número de acciones permitidas para que el bucle del juego sepa cuántas veces preguntar al jugador
        return self.acciones_permitidas

# ========================================== # Separador visual final
# INICIALIZACIÓN DEL SISTEMA                 # Título descriptivo
# ========================================== # Separador visual final

# Instanciamos el motor de clima globalmente para que ambos jugadores sufran las mismas condiciones
motor_clima = GestorClima()

In [4]:
# ============================================================================ # Encabezado estético de la celda
# FASE 4 REVISADA: TABLERO GRÁFICO A COLOR CON MOTOR DE EMOJIS                  # Título funcional de la sección
# ============================================================================ # Cierre del encabezado

import random # Importamos la librería random para posicionar la flota del juego

class TableroReal: # Definimos la clase TableroReal que controlará la matriz visual
    def __init__(self, nombre_jugador): # Constructor de la clase
        self.propietario = nombre_jugador # Guardamos el nombre del dueño del tablero
        self.matriz = [["~" for _ in range(10)] for _ in range(10)] # Matriz de 10x10 inicializada con agua básica

        # Diccionario con los tamaños oficiales de cada embarcación
        self.tamanios = {"portaaviones": 5, "ingeniero": 4, "pesquero": 3, "inteligencia": 2}
        # Diccionario de identificadores internos para el backend del código
        self.simbolos = {"portaaviones": "P", "ingeniero": "E", "pesquero": "S", "inteligencia": "I"}

        # NUEVO MOTOR GRÁFICO: Mapeamos los caracteres internos a emojis a color para la pantalla
        self.skins = {
            "~": "🟦",  # Mar abierto (Azul)
            "P": "🚢",  # Portaaviones Gigante
            "E": "🛠️",  # Barco de Ingenieros
            "S": "🎣",  # Barco Pesquero Comercial
            "I": "📡",  # Barco de Inteligencia Radar
            "X": "💥",  # Impacto / Explosión (Fuego)
            "O": "⚪"   # Disparo fallido / Agua
        }

        self.ubicar_barcos_aleatorio() # El bot usa este método, el humano lo sobrescribirá en la Fase 5

    def ubicar_barcos_aleatorio(self): # Método para rellenar el mapa automáticamente (usado por la IA)
        for tipo in self.tamanios.keys(): # Recorremos cada tipo de barco disponible
            colocado = False # Bandera de control para el bucle de posicionamiento
            while not colocado: # Se repite hasta encontrar un espacio legal en el mapa
                fila = random.randint(0, 9) # Elige fila al azar
                col = random.randint(0, 9) # Elige columna al azar
                orientacion = random.choice(["H", "V"]) # Elige si ponerlo horizontal o vertical
                if self.validar_e_insertar(tipo, fila, col, orientacion): # Intenta meterlo en la matriz
                    colocado = True # Rompe el bucle si tuvo éxito

    def dibujar_radar(self, oculto=False): # Método encargado de imprimir el mapa con los emojis coloridos
        titulo = f"📡 RADAR DEL ENEMIGO (Oculto)" if oculto else f"🗺️ TU FLOTA ({self.propietario})" # Define el encabezado
        print(f"\n{titulo}") # Imprime el título en la consola
        print("    A  B  C  D  E  F  G  H  I  J") # Imprime la guía alfabética de columnas
        print("  -------------------------------") # Separador estético superior

        for i in range(10): # Bucle que recorre las 10 filas del tablero
            fila_visual = [] # Lista temporal para armar los emojis de la fila actual
            for celda in self.matriz[i]: # Recorremos cada celda de la fila horizontal
                if oculto and celda in ["P", "E", "S", "I"]: # Condicional: Si es el mapa enemigo y hay un barco vivo...
                    fila_visual.append(self.skins["~"]) # ...lo enmascaramos pintando agua azul '🟦' para ocultarlo
                else: # Si es tu mapa o si es un impacto 'X' o fallo 'O'
                    fila_visual.append(self.skins[celda]) # Traduce el carácter interno al emoji correspondiente

            num_fila = str(i + 1).rjust(2) # Formatea el número lateral (1-10) para mantener la alineación perfecta
            print(f"{num_fila}| {' '.join(fila_visual)} |") # Imprime la línea completa armada con los emojis espaciados
        print("  -------------------------------") # Separador estético inferior

    def validar_e_insertar(self, tipo_barco, fila_inicio, col_inicio, orientacion): # Valida espacio geométrico en la matriz
        tam = self.tamanios[tipo_barco] # Obtiene la longitud del barco
        simbolo = self.simbolos[tipo_barco] # Obtiene la letra interna del barco

        if orientacion == "H" and (col_inicio + tam > 10): # Valida si el barco se sale por el borde derecho
            return False # Rechaza la posición
        if orientacion == "V" and (fila_inicio + tam > 10): # Valida si el barco se sale por el borde inferior
            return False # En el fondo del mapa, rechaza

        for i in range(tam): # Bucle para escanear colisiones con barcos ya existentes
            f = fila_inicio + (i if orientacion == "V" else 0) # Calcula la fila de la casilla actual
            c = col_inicio + (i if orientacion == "H" else 0) # Calcula la columna de la casilla actual
            if self.matriz[f][c] != "~": # Condicional: Si la casilla ya está ocupada por otra letra...
                return False # Detectó choque, rechaza la posición

        for i in range(tam): # Si pasó todas las pruebas anteriores, procede a escribirlo en la matriz
            f = fila_inicio + (i if orientacion == "V" else 0) # Calcula coordenada Y
            c = col_inicio + (i if orientacion == "H" else 0) # Calcula coordenada X
            self.matriz[f][c] = simbolo # Escribe la letra del barco en la celda
        return True # Retorna True confirmando el despliegue exitoso

In [5]:
# ============================================================================ # Encabezado estético que marca el inicio del bloque.
# FASE 5 DEFINITIVA: IA MODO CAZADOR Y PANEL DE INTELIGENCIA PERSISTENTE # Título que describe la funcionalidad de esta versión del código.
# ============================================================================ # Cierre del encabezado estético.

import time # Importa la librería 'time' para poder hacer pausas temporales (ej. para que la IA simule pensar).
import random # Importa 'random' para generar números aleatorios (coordenadas, clima, sorteos).
import ipywidgets as widgets # Importa 'ipywidgets' para crear la interfaz visual (botones, cajas de texto, etc.).
from IPython.display import display, clear_output, HTML # Importa herramientas de IPython para mostrar cosas, limpiar la pantalla y renderizar HTML.
import contextlib # Importa 'contextlib' para manejar contextos especiales, como suprimir impresiones en la consola.
import io # Importa 'io' para crear salidas de texto temporales en memoria (ayuda a ocultar texto no deseado).

def traducir_coordenada(texto): # Define la función que convierte un texto ("A5") en índices de matriz (0, 4).
    try: # Inicia un bloque try-except para que no colapse si el usuario mete texto basura.
        texto = str(texto).upper().strip() # Convierte la entrada a texto, lo pasa a mayúsculas y le quita espacios a los lados.
        if len(texto) < 2: return None # Verifica que la entrada tenga al menos 2 caracteres; si no, devuelve None (error).
        letra, numero = texto[0], texto[1:] # Extrae la letra (primer carácter) y el número (del segundo carácter en adelante).
        columnas_mapa = {"A":0, "B":1, "C":2, "D":3, "E":4, "F":5, "G":6, "H":7, "I":8, "J":9} # Diccionario que traduce las letras a números (0 a 9).
        if letra not in columnas_mapa or not numero.isdigit(): return None # Si la letra es falsa o el resto no es un número, devuelve None.
        f_idx, c_idx = int(numero) - 1, columnas_mapa[letra] # Calcula el índice de la fila (número-1) y saca el de columna del diccionario.
        return (f_idx, c_idx) if (0 <= f_idx < 10 and 0 <= c_idx < 10) else None # Devuelve (fila, columna) si están dentro del mapa 10x10, sino devuelve None.
    except: # Si cualquier paso del 'try' generó un error de código...
        return None # ...lo atrapa silenciosamente y devuelve None.

class MotorJuegoWidgets: # Define la clase principal que controla la lógica, reglas y la interfaz gráfica del juego.
    def __init__(self, humano, maquina, mapa_h, mapa_m, clima): # Constructor: Se ejecuta al crear el juego. Recibe jugadores, mapas y clima.
        self.humano = humano # Asigna el jugador humano (viene de Fases anteriores) a una variable local.
        self.maquina = maquina # Asigna la IA (viene de Fases anteriores) a una variable local.
        self.mapa_h = mapa_h # Asigna el tablero visual del humano.
        self.mapa_m = mapa_m # Asigna el tablero visual de la IA.
        self.clima = clima # Asigna el generador de clima al juego.

        self.humano.creditos = 750 # REGLA: Establece los fondos iniciales del humano en 750 (parche de balance).
        self.maquina.creditos = 750 # REGLA: Establece los fondos iniciales de la IA en 750.
        for b in self.humano.flota + self.maquina.flota: # Recorre todos los barcos de ambas flotas juntos.
            b.vida_actual = b.vida_max # REGLA: Fuerza a que cada barco comience con el 100% de su vida para evitar bugs médicos.

        self.fase = "COLOCACION" # Variable de estado: Define que el juego inicia en la etapa de colocar barcos.
        self.barcos_por_colocar = ["portaaviones", "ingeniero", "pesquero", "inteligencia"] # Orden lógico en el que los barcos van a ser colocados en el tablero.
        self.barco_actual_idx = 0 # Índice que rastrea qué barco (de la lista anterior) se está colocando en este momento.
        self.acciones_restantes = 0 # Inicializa en 0 la variable que cuenta cuántos disparos/acciones te quedan.
        self.acciones_turno_anterior = None # Inicializa variable para guardar el historial de acciones.
        self.ronda = 1 # Define que el contador general de rondas arranca en 1.

        self.log_ia = [] # Lista vacía que guardará los textos de todo lo que haga la IA en su turno.
        self.mensaje_sistema = "Comandante, a la espera de sus órdenes." # Mensaje de estado estándar del panel.
        self.mensaje_radar = "" # Almacena las coordenadas reveladas por el radar sigiloso para mostrarlas.
        self.radar_turnos_vida = 0 # Define cuántos turnos/rondas durará vivo el reporte del radar en pantalla.

        self.info_armas = { # Base de datos económica y de estadísticas para las armas.
            'cañon': {'costo': 500, 'danio': 30}, # Cañón: vale 500 CR, pega 30 HP.
            'artilleria': {'costo': 750, 'danio': 40}, # Artillería: vale 750 CR, pega 40 HP.
            'bombardeo': {'costo': 900, 'danio': 50}, # Bombardeo: vale 900 CR, pega 50 HP.
            'torpedo': {'costo': 1500, 'danio': 60} # Torpedo: vale 1500 CR, pega 60 HP.
        } # Fin del diccionario de armas.

        self.contenedor_principal = widgets.VBox() # Crea un contenedor vertical maestro para apilar todos los elementos gráficos.
        self.output_pantalla = widgets.Output() # Crea un área de dibujo ("lienzo") donde se imprimirá el juego sin parpadeos.

        self.txt_coord = widgets.Text(placeholder="Ej: C4", description="Casilla:", layout=widgets.Layout(width='150px')) # Crea la caja blanca donde el usuario teclea (A5, B9).
        self.drop_orientacion = widgets.Dropdown(options=[('Horizontal', 'H'), ('Vertical', 'V')], description='Dirección:', layout=widgets.Layout(width='180px')) # Crea el menú para poner el barco acotado o parado.

        self.btn_confirmar = widgets.Button(description="Desplegar Barco", button_style='success', icon='ship') # Crea el botón verde de "Desplegar Barco".
        self.btn_aleatorio = widgets.Button(description="Despliegue Aleatorio", button_style='info', icon='random') # Crea el botón azul claro para "Despliegue Automático".

        self.btn_atacar = widgets.Button(description="Disparar", button_style='danger', icon='crosshairs') # Crea el botón rojo principal para Disparar.
        self.btn_sigilo = widgets.Button(description="Radar Sigiloso", button_style='primary', icon='eye') # Crea el botón azul para la habilidad especial de Inteligencia.
        self.btn_reparar = widgets.Button(description="Reparar", button_style='success', icon='wrench') # Crea el botón verde para curar barcos (Ingeniero).
        self.btn_pesquero = widgets.Button(description="+ Créditos", button_style='info', icon='money-bill') # Crea el botón azul celeste para ganar plata (Pesquero).
        self.drop_tienda = widgets.Dropdown(options=[('Cañón 30 DMG (500)', 'cañon'), ('Artillería 40 DMG (750)', 'artilleria'), ('Bombardeo 50 DMG (900)', 'bombardeo'), ('Torpedo 60 DMG (1500)', 'torpedo')], description='Tienda:') # Crea el menú desplegable de la tienda de armas con sus stats.
        self.btn_comprar = widgets.Button(description="Comprar", button_style='warning', icon='shopping-cart') # Crea el botón amarillo/naranja para efectuar compras.
        self.btn_pasar = widgets.Button(description="Pasar Turno", button_style='', icon='fast-forward') # Crea el botón sin color para ceder o saltar turno.

        self.txt_enter = widgets.Text(placeholder="[ HAZ CLIC AQUÍ Y PRESIONA ENTER PARA AVANZAR ]", layout=widgets.Layout(width='400px')) # Crea una caja de texto larga que actúa como el freno/pausa para avanzar ronda.

        self.btn_confirmar.on_click(self.procesar_colocacion) # Conecta el click del botón Desplegar con el código que pone barcos.
        self.btn_aleatorio.on_click(self.procesar_aleatorio_h) # Conecta el botón aleatorio con su código correspondiente.
        self.btn_atacar.on_click(self.procesar_ataque_h) # Conecta el botón Disparar con la función principal de ataque.
        self.btn_sigilo.on_click(self.procesar_sigilo_h) # Conecta el botón del Radar con la lógica de espiar coordenadas.
        self.btn_reparar.on_click(self.procesar_reparacion_h) # Conecta el botón de Reparar con la función que suma HP y borra X.
        self.btn_pesquero.on_click(self.procesar_pesquero_h) # Conecta el botón del Pesquero a la función de recaudar dinero.
        self.btn_comprar.on_click(self.procesar_compra_h) # Conecta el botón Comprar a la lógica que verifica si hay dinero e inyecta balas.
        self.btn_pasar.on_click(self.procesar_pasar_turno) # Conecta el botón Pasar Turno con la lógica que manda todo a 0 acciones.
        self.txt_enter.on_submit(self.procesar_siguiente_ronda) # Conecta el apretar 'Enter' sobre la caja con la función para empezar la siguiente ronda.

        self.mapa_m.matriz = [["~" for _ in range(10)] for _ in range(10)] # Resetea la matriz 10x10 de la máquina, llenándola de agua ("~").
        self.mapa_m.ubicar_barcos_aleatorio() # Le ordena a la máquina (ocultamente) que tire sus barcos en su mapa al azar.
        self.mapa_h.matriz = [["~" for _ in range(10)] for _ in range(10)] # Resetea el mapa del humano en agua ("~").

        self.actualizar_controles() # Llama a la función que renderiza qué botones se deben ver al inicio (colocación).
        self.actualizar_pantalla() # Ejecuta el primer dibujo total de la interfaz.

    def obtener_html_reglas(self): # Función dedicada exclusivamente a devolver el texto visual del manual.
        return """ # Retorna un String multi-línea (para facilitar el uso de código HTML dentro de Python).
        <div style='background-color: #0b192c; color: #ecf0f1; padding: 15px; border-radius: 8px; margin-bottom: 15px; border: 1px solid #34495e;'> # Abre un panel azul oscuro, con texto claro y bordes redondeados.
            <h3 style='margin-top: 0; color: #f1c40f;'>📜 MANUAL TÁCTICO DE OPERACIONES</h3> # Título principal en color dorado.
            <p><b>1. SISTEMA CLIMÁTICO Y TURNOS:</b> El clima de la zona cambia cada ronda. Define tus acciones (1 a 3) y desata eventos que afectan tu flota.</p> # Párrafo de la regla climática.
            <p><b>2. HABILIDADES DE LA FLOTA (¡Si un barco es hundido, pierdes su habilidad permanentemente!):</b></p> # Párrafo de introducción a los barcos.
            <ul style='margin-bottom: 10px; padding-left: 20px; font-size: 14px;'> # Abre una lista con viñetas.
                <li><b>Portaaviones (5 Casillas):</b> Te permite comprar y disparar <i>Artillería</i> (40 DMG) y <i>Bombardeos</i> (50 DMG).</li> # Regla: Portaaviones y armas pesadas.
                <li><b>Ingeniero (4 Casillas):</b> Habilita la compra de <i>Torpedos</i> (60 DMG) y la acción táctica de <i>Reparar</i>.</li> # Regla: Ingeniero, curación y torpedos.
                <li><b>Pesquero (3 Casillas):</b> Activa la opción <i>+ Créditos</i>. Si es hundido, rescatarás de 1 a 3 cañones gratis por ronda.</li> # Regla: Pesquero, dinero y pasiva de balas.
                <li><b>Inteligencia (2 Casillas):</b> Activa el <i>Radar Sigiloso</i>. Consume <b>todos los turnos</b>, pero te revela 5 coordenadas enemigas (1 real y 4 falsas).</li> # Regla: Barco espía y sigilo.
            </ul> # Cierra la lista de viñetas.
            <p style='margin-top: 10px; margin-bottom: 0;'><b>3. ESTRATEGIA DE REPARACIÓN:</b> La opción "Reparar" recupera <b>30 HP</b> y oculta una casilla dañada de tu mapa. <span style='color: #e74c3c; font-weight: bold;'>¡RIESGO!</span> Al reparar no haces daño, y el enemigo ya conoce tu ubicación.</p> # Párrafo de advertencia de la reparación.
        </div> # Cierra el panel de reglas principal.
        """ # Cierre del String multi-línea.

    def actualizar_controles(self): # Lógica de visibilidad: Escoge qué botones agrupar e imprimir según la fase del juego.
        if self.fase == "COLOCACION": # Si estamos acomodando la flota inicial...
            controles = widgets.HBox([self.txt_coord, self.drop_orientacion, self.btn_confirmar, self.btn_aleatorio]) # Muestra en horizontal la caja de texto, dirección y los dos botones verdes/azules de confirmar.
        elif self.fase == "COMBATE": # Si estamos en la guerra (turno tuyo)...
            fila_ataque = widgets.HBox([self.txt_coord, self.btn_atacar, self.btn_sigilo]) # Crea una fila superior con: texto, Atacar y Sigilo.
            fila_utilidad = widgets.HBox([self.btn_reparar, self.btn_pesquero, self.btn_pasar]) # Crea fila media con botones de soporte: Reparar, Pesquero, Pasar.
            fila_tienda = widgets.HBox([self.drop_tienda, self.btn_comprar]) # Crea fila inferior con armería: Menú y Botón comprar.
            controles = widgets.VBox([fila_ataque, fila_utilidad, fila_tienda]) # Agrupa (VBox) las tres filas en un solo bloque.
        elif self.fase == "TURNO_IA": # Si es el turno de la máquina (protección contra trampas)...
            controles = widgets.HTML("<h3 style='color:#e67e22; text-align:center;'>🤖 Bloqueo de sistema: La IA está operando...</h3>") # Bloquea todo control y muestra letrero narrativo de espera.
        elif self.fase == "PAUSA_RONDA": # Si se requiere pausa para revisar la información...
            controles = widgets.VBox([self.txt_enter], layout=widgets.Layout(align_items='center')) # Muestra únicamente la caja de input para el ENTER y la centra.
        elif self.fase == "FIN": # Si se detectó una victoria o derrota...
            controles = widgets.HTML("<h2>Juego Finalizado. Reinicia la celda para volver a jugar.</h2>") # Muestra mensaje de fin.

        self.contenedor_principal.children = [self.output_pantalla, controles] # Actualiza el contenedor padre inyectando los controles decididos debajo de la pantalla de gráficos.

    def verificar_victoria(self): # Función global de muerte súbita para chequear si alguien ya ganó.
        vivos_h = sum(1 for b in self.humano.flota if b.vivo) # Cuenta cuántos barcos tienen la propiedad 'vivo' en la lista del Humano.
        vivos_m = sum(1 for b in self.maquina.flota if b.vivo) # Cuenta cuántos barcos tienen 'vivo' en la Máquina.

        if vivos_h == 0: # Si al Humano no le queda nadie...
            self.fase = "FIN" # Transiciona forzosamente a fase FIN.
            self.mensaje_sistema = "💀 GAME OVER. La IA ha destruido toda tu flota." # Carga texto fatídico.
            self.actualizar_controles() # Dispara bloqueo de pantalla inmediata.
            return True # Devuelve que hubo Game Over.
        elif vivos_m == 0: # Si la IA se quedó a cero...
            self.fase = "FIN" # Activa fase FIN.
            self.mensaje_sistema = "🏆 ¡VICTORIA MAGNA! Has erradicado la flota enemiga del océano." # Declara victoria al humano.
            self.actualizar_controles() # Bloquea controles y tira letrero.
            return True # Devuelve que hubo Game Over.
        return False # Si la función llega acá, ambos tienen vivos, devuelve Falso para que el juego continúe normal.

    def aplicar_niveles_flota(self, jugador): # Función RPG de Subida de Nivel. Revisa un jugador y escala stats de su flota.
        for b in jugador.flota: # Para cada barco en la lista del jugador elegido...
            b.nivel += 1 # Sube el escalafón lógico de nivel en 1.
            if b.tipo == 'portaaviones': # Comprueba la clase del barco (si es el grande).
                if b.nivel == 2: b.vida_max = 125; b.vida_actual += 25 # Si llega al 2: suelta el cap de vida a 125 y le regala 25 de vida gratis.
                elif b.nivel == 3: b.vida_max = 150; b.vida_actual += 25 # Si llega al 3: suelta cap a 150 y le regala 25.
            elif b.tipo == 'inteligencia': # Comprueba si es el pequeñín espía.
                if b.nivel == 3: b.vida_max = 50; b.vida_actual += 10 # Al nivel 3 se vuelve fuerte: max vida sube a 50 y cura 10.
            elif b.tipo == 'ingeniero': # Comprueba si es el de soporte de cura.
                b.vida_max = 80  # Mantiene su tope estático a 80 por decisión del reglamento tuyo.
            elif b.tipo == 'pesquero': # Comprueba si es el generador económico.
                if b.nivel == 2: b.vida_max = 75; b.vida_actual += 15; b.creditos_generados = 650 # Sube vida a 75, cura 15, y sube salario a 650.
                elif b.nivel == 3: b.creditos_generados = 750 # Ya no sube vida, solo incrementa el salario a 750 (max).

    def renderizar_panel_html(self): # Generador del bloque gráfico para mostrar información durante el combate.
        estilo_css = "<style> pre { font-size: 16px !important; line-height: 1.3 !important; font-family: 'Courier New', monospace; } </style>" # Forzador CSS para que el radar ascii (----) no se descuadre en IPython.
        display(HTML(estilo_css)) # Envía el comando de estilo a la celda.

        if self.ronda == 1: # Filtro condicional: Si estamos estrenando el combate.
            display(HTML(self.obtener_html_reglas())) # Manda a imprimir la caja de las reglas por única vez.

        estado_barcos = "" # Inicializador del texto largo con la vida de cada barco.
        for b in self.humano.flota: # Repasa 4 veces tu flota para llenar la lista.
            icono = "🟢" if b.vivo else "🔴" # Decide un semáforo visual rápido: Verde o Rojo si 'vivo' está true o false.
            hp = f"{b.vida_actual}/{b.vida_max} HP" if b.vivo else "HUNDIDO" # Da la estadística actual (ej 75/100 HP) o grita HUNDIDO si es rojo.
            estado_barcos += f"&nbsp;&nbsp;{icono} <b>{b.nombre.capitalize()}</b> (Nv {b.nivel}): {hp}<br>" # Ensambla un reglón y lo suma al acumulador de HTML, inyectando nombre y nivel.

        clima_limpio = str(self.clima.evento_actual).split('\n')[-1][:80] # Extrae solo la parte de abajo de tu consola del clima para que encaje bonito (evita acumular textos y saltos '\n').

        html_panel = f""" # Empieza la concatenación gigante del marco gráfico HTML f-string.
        <div style='font-family: "Segoe UI", Tahoma, sans-serif; max-width: 800px;'> # Abre el marco y asigna fuente elegante.

            <div style='background-color: #2c3e50; color: white; padding: 15px; border-radius: 8px; margin-bottom: 10px; text-align: center; border: 2px solid #3498db;'> # Tablilla superior.
                <h2 style='margin: 0; color: #00ffcc;'>🌊 RONDA DE COMBATE #{self.ronda} 🌊</h2> # Cabecera azul aqua que cambia según variable self.ronda.
                <h4 style='margin: 5px 0 0 0;'>🌤️ Clima: <span style='color:#f1c40f;'>{clima_limpio.upper()}</span> | ⚡ Acciones Restantes: <span style='color:#e74c3c; font-size: 20px;'>{self.acciones_restantes}</span></h4> # Dicta en letras el evento filtrado del clima y el número en rojo de los tiros que restan.
            </div> # Cierra panel azul superior.

            <div style='background-color: #1e272e; color: white; padding: 15px; border-radius: 8px; margin-bottom: 15px; border-left: 5px solid #0be881;'> # Panel secundario de Inventario.
                <h3 style='margin: 0 0 10px 0; color: #0be881;'>📊 PANEL DEL COMANDANTE</h3> # Título comandante.
                <b>💰 Billetera:</b> {self.humano.creditos} Créditos<br> # Línea 1: Consulta tu billetera interna actual y la pinta.
                <b>📦 Inventario Armería:</b> Torpedos (60 DMG): {int(self.humano.municion['torpedo'])} | Cañones (30 DMG): {int(self.humano.municion['cañon'])} | Artillería (40 DMG): {int(self.humano.municion['artilleria'])} | Bombardeos (50 DMG): {int(self.humano.municion['bombardeo'])}<br> # Muestra la cantidad guardada en el diccionario de tus 4 tipos de balas.
                <b>🚢 Estado de la Flota:</b><br> # Apartado para el listado HP.
                {estado_barcos} # Inyecta la variable que pre-procesamos arriba llena de luces de semáforo verdes.
                <hr style='border-color: #485460;'> # Línea horizontal separadora.
                <b style='color: #0fb9b1;'>🔔 REPORTE DE SISTEMA:</b> {self.mensaje_sistema} # Variable crítica de texto donde la consola te avisa qué está pasando si te equivocas.
            </div> # Cierra bloque Comandante.
        """ # Frena la variable.

        # PANEL DEL RADAR SIGILOSO (Persistente)
        if self.mensaje_radar != "": # Si activaste la Inteligencia, esta variable tiene texto (la coordenada revelada)...
            html_panel += f""" # Acopla otra tajada HTML de cajón azul de espionaje.
            <div style='background-color: #2980b9; color: white; padding: 15px; border-radius: 8px; margin-bottom: 15px; border-left: 5px solid #3498db;'> # Diseño del cajón.
                <h4 style='margin: 0 0 5px 0; color: #00ffcc;'>🕵️‍♂️ REPORTE DE INTELIGENCIA ACTIVO:</h4> # Título secreto.
                <p style='margin: 0;'>{self.mensaje_radar}</p> # Inyecta la variable secreta con las 5 coordenadas extraídas.
            </div> # Cierra cajón de espionaje.
            """ # Cierra variable a acoplar.

        if self.log_ia: # Chequea si el arreglo/lista donde escribe la Máquina tiene algo escrito (cuando es su turno y juega).
            html_panel += "<div style='background-color: #440000; color: white; padding: 15px; border-radius: 8px; margin-bottom: 15px; border-left: 5px solid #ff3838;'>" # Cajón carmesí para el enemigo.
            html_panel += "<h3 style='margin: 0 0 10px 0; color: #ff3838;'>🚨 REPORTE DEL TURNO ENEMIGO:</h3>" # Título de log de IA.
            for accion in self.log_ia: # Itera leyendo en orden todas las jugadas sucias de la máquina.
                html_panel += f"• {accion}<br>" # Por cada una, añade una viñeta circular en HTML al listado.
            html_panel += "</div>" # Cierra bloque carmesí.

        html_panel += "</div>" # Cierre de etiqueta global invisible.
        display(HTML(html_panel)) # Le dice al Output del notebook que suelte y pinte la obra maestra final a pantalla.

    def actualizar_pantalla(self): # Orquestador maestro visual que sabe qué orden dibujar y limpia la basura.
        with self.output_pantalla: # Entra en el contexto exclusivo de nuestra pantalla dedicada "lienzo".
            clear_output(wait=True) # Elimina por completo lo que estabas viendo el milisegundo pasado, evitando titilar.
            if self.fase == "COLOCACION": # Pregunta a la variable: ¿Seguimos acomodando barcos?
                display(HTML("<h2 style='font-family: sans-serif; color: #3498db;'>⚓ FASE DE DESPLIEGUE</h2>")) # Si sí, tira letrero inicial de Despliegue.
                display(HTML(self.obtener_html_reglas())) # Pinta el manual en toda la cara del usuario.

                self.mapa_h.dibujar_radar(oculto=False) # Dibuja la matriz Ascii humano en pantalla sin modo niebla (oculto falso).
                barco_nombre = self.barcos_por_colocar[self.barco_actual_idx] # Interroga al índice cuál te toca ubicar.
                tam = self.mapa_h.tamanios[barco_nombre] # Deduce su tamaño según el diccionario del barco.
                display(HTML(f"<h4 style='font-family: sans-serif;'>🚢 Posiciona tu <b>{barco_nombre.upper()}</b> (Ocupa {tam} casillas).</h4>")) # Te pide que lo acomodes usando HTML y bold.
                if self.mensaje_sistema != "Comandante, a la espera de sus órdenes.": # Si hay un reporte distinto (o sea, un error de choque)...
                    print(f"⚠️ {self.mensaje_sistema}") # ...lo imprime en advertencia amarilla.
            elif self.fase in ["COMBATE", "TURNO_IA", "PAUSA_RONDA", "FIN"]: # Si por el contrario la partida arrancó o ya terminó...
                self.renderizar_panel_html() # Pinta el bloque HTML gigante de arriba.
                print("\n") # Un espacio para que respiren visualmente el panel de los mapas.
                self.mapa_m.dibujar_radar(oculto=True) # Dibuja el del enemigo EN MODO NIEBLA (no ves sus letras, solo tus tiros).
                self.mapa_h.dibujar_radar(oculto=False) # Dibuja tu radar personal transparente (donde sufres los tiros).

    def procesar_colocacion(self, b): # EVENTO DEL BOTÓN VERDE "Desplegar Barco" manual.
        barco_nombre = self.barcos_por_colocar[self.barco_actual_idx] # Lee el índice actual.
        indices = traducir_coordenada(self.txt_coord.value) # Manda al decodificador a romper el "B4" a números (1, 3).
        orientacion = self.drop_orientacion.value # Averigua la letra H/V seleccionada del botón menú.

        if indices is None: # Si el traductor dictó 'error'...
            self.mensaje_sistema = "❌ Error: Coordenada inválida (Ej: B3)." # Rebota al usuario y tira error a la consola.
        else: # Si sí sonó lógico...
            fila, col = indices # Desempaqueta las coordenadas crudas.
            if self.mapa_h.validar_e_insertar(barco_nombre, fila, col, orientacion): # Ordena al mapa a través de su función validora ver si entra sin chocar ni salirse; Si retorna TRUE:
                self.barco_actual_idx += 1 # Agrega el siguiente a la lista, pasas de nivel.
                self.txt_coord.value = "" # Borra lo que tenías escrito porque ya sirvió.
                self.mensaje_sistema = "Comandante, a la espera de sus órdenes." # Relaja al sistema a su estatus pacífico.
                if self.barco_actual_idx >= len(self.barcos_por_colocar): # Pregunta de oro: Si el contador llegó a 4 (acabamos los barcos)...
                    self.mapa_h_original = [f[:] for f in self.mapa_h.matriz] # Guarda una fotografía matriz (copia profunda) de TODO el campo base Humano.
                    self.mapa_m_original = [f[:] for f in self.mapa_m.matriz] # Guarda foto igualita de las letras IA para el futuro motor Curativo.

                    self.fase = "COMBATE" # Transición de estado definitivo a guerra bélica.
                    with io.StringIO() as buf, contextlib.redirect_stdout(buf): # Filtro quirúrgico para callar impresiones terminales sucias de la lógica previa del clima.
                        self.acciones_restantes = self.clima.sortear_clima() # Obliga al algoritmo a sortear eventos por fin (1,2 o 3 turnos para ti y la IA).
                    self.acciones_turno_anterior = self.acciones_restantes # Lo sube al historial.
                    self.actualizar_controles() # Oculta botones de colocar, despliega botones bélicos.
            else: # Si tu barco chocó con el portaaviones o se salió por F10:
                self.mensaje_sistema = "❌ Posición ilegal. El barco NO CABE (se sale del mapa) o choca con otro." # Explica duramente por qué el código te rebotó.
        self.actualizar_pantalla() # Re-carga todo para mostrar tu progreso.

    def procesar_aleatorio_h(self, b): # EVENTO DEL BOTÓN AZUL CLARO "Aleatorio" de arranque rápido.
        while self.barco_actual_idx < len(self.barcos_por_colocar): # Bucle infinito mientras te sigan faltando barcos por colocar...
            barco_nombre = self.barcos_por_colocar[self.barco_actual_idx] # Ve nombrando el que toca.
            colocado = False # Interruptor lógico de control de estancamiento.
            while not colocado: # Bucle de fuerza bruta para probar ubicaciones incesantemente...
                f, c = random.randint(0, 9), random.randint(0, 9) # Cierra los ojos y tira dardo para matriz.
                ori = random.choice(['H', 'V']) # Tira moneda.
                if self.mapa_h.validar_e_insertar(barco_nombre, f, c, ori): # Si el dardo le atinó a agua sana e insertó el barco...
                    colocado = True # Desactiva el mini bucle de fuerza bruta de posición.
                    self.barco_actual_idx += 1 # Avanza a ordenar posicionar el otro barco.

        self.txt_coord.value = "" # Borra caja texto inútil.
        self.mensaje_sistema = "Comandante, flota desplegada automáticamente." # Envía aviso de éxito.
        self.mapa_h_original = [f[:] for f in self.mapa_h.matriz] # Guarda fotocopia intacta matriz H para el motor Médico.
        self.mapa_m_original = [f[:] for f in self.mapa_m.matriz] # Fotocopia M intacta para M motor médico.

        self.fase = "COMBATE" # Entramos a fase agresiva.
        with io.StringIO() as buf, contextlib.redirect_stdout(buf): # Bloque mágico silencioso de impresiones de basura en pantalla.
            self.acciones_restantes = self.clima.sortear_clima() # Regala los primeros turnos climáticos.
        self.acciones_turno_anterior = self.acciones_restantes # Subir al guardado log.
        self.actualizar_controles() # Actualiza fila de botones para ti (disparo, curar).
        self.actualizar_pantalla() # Dispara visualmente la FASE COMBATE inicial.

    def procesar_siguiente_ronda(self, sender): # EVENTO DE PAUSA ENTER (Sucede al dar Enter al finalizar IA).
        self.txt_enter.value = ""  # Borra el texto/espacio en blanco que enviaste como tecla Enter para limpiar caja.
        self.ronda += 1 # Hace avanzar el número global maestro de 'Rondas de Combate'.

        # Lógica de persistencia del radar del panel azul.
        if hasattr(self, 'radar_turnos_vida') and self.radar_turnos_vida > 0: # Pregunta si el contador está inicializado y sigue vivo...
            self.radar_turnos_vida -= 1 # Si es así, quémale su vida.
            if self.radar_turnos_vida <= 0: # Si al quemarlo llega o cruza el 0...
                self.mensaje_radar = "" # Mata la variable y hace desaparecer el panel azul de la siguiente pantalla redibujada.

        extra_msg = "" # Variable en limpio para apendar los mil avisos extras de pasivas.
        pesq_h = [b for b in self.humano.flota if b.tipo == 'pesquero'][0] # Verifica estado vital de tu pesquero comercial.
        if not pesq_h.vivo: # REGLA DE JUEGO (PASIVA): Si la IA te hundió el pesquero...
            balas = random.randint(1, 3) # Ejecuta suerte y saca 1 a 3 balas de auxilio.
            self.humano.municion['cañon'] += balas # Inyecta las balas de cañón directo al diccionario inventario Humano.
            extra_msg = f" 📦 Los restos del Pesquero rescataron {balas} Cañones." # Guarda el mensaje inmersivo para decírtelo.

        if (self.ronda - 1) % 8 == 0: # MOTOR RPG NIVELES: (8, 16, 24). Resta 1 (por inicio ronda 1), Modulo 8. ¿Es 0? (División exacta por 8).
            self.aplicar_niveles_flota(self.humano) # Dispara función aumetadora estadísticas tuyas.
            self.aplicar_niveles_flota(self.maquina) # Sube la IA para que siga compitiendo.
            self.mensaje_sistema = f"🌟 ¡SUBIDA DE NIVEL! Barcos al Nv {(self.ronda-1)//8 + 1}." + extra_msg # Avisa la epifanía RPG en tu panel verde.
        else: # Si no dividía entre 8 (turno gris estándar)...
            self.mensaje_sistema = "Es tu turno Comandante." + extra_msg # Informa tu turno estándar pero apendiendo posibles balitas de pasivas.

        with io.StringIO() as buf, contextlib.redirect_stdout(buf): # Callando el molesto 'print' que trae la función vieja de clima en terminal normal.
            self.acciones_restantes = self.clima.sortear_clima() # Pide nuevos rayos o calmas a la madre naturaleza (Turnos al azar).

        self.log_ia = [] # Vacía el registro de balas y decisiones de la IA porque su ronda ya fue purgada (Amanece un nuevo día).
        self.fase = "COMBATE" # Elimina la variable de bloqueo (PAUSA RONDA) pasándola a modo guerra normal de tu turno.
        self.actualizar_controles() # Refresca tus botoneras para dejarlas de colorines lista al ataque.
        self.actualizar_pantalla() # Dibuja una vez más toda la interfaz ya limpia.

    def restaurar_casilla(self, mapa_modificado, mapa_original, tipo_barco): # MOTOR CURATIVO DE INTERFAZ: Borra visualmente las cruces rojas.
        letras = {"portaaviones":"P", "ingeniero":"E", "pesquero":"S", "inteligencia":"I"} # Define diccionario traductor de clases a símbolos Ascii.
        letra_buscada = letras[tipo_barco] # Define objetivo (ej, E si curaron ingeniero).
        candidatos = [] # Crea lista por si el barco es inmenso y tiene 5 casillas rotas.
        for f in range(10): # Repasa todo el plano horizontal...
            for c in range(10): # ...y todo el vertical.
                if mapa_modificado.matriz[f][c] == "X" and mapa_original[f][c] == letra_buscada: # Si hay X pero en la fotocopia guardada era una letra legal...
                    candidatos.append((f, c)) # Es un pedazo de mi barco herido; añadirlo al arreglo candidato.
        if candidatos: # Si hubo al menos un pedazo hallado herido...
            f_r, c_r = random.choice(candidatos) # Dispara aleatoriedad y selecciona un cuadrante solo de esa selección filtrada (curación visual justa).
            mapa_modificado.matriz[f_r][c_r] = letra_buscada # Inscribe de vuelta a su letra vieja ("E", "P") sanando la marca roja "X".

    def descontar_accion(self): # SUPERVISOR DE FLUJO DE CAJA DE TIEMPO BÉLICO (Determina vida del loop y salta de H->M).
        if self.verificar_victoria(): # Llamado a juez de muerte global. ¿Alguien mató a alguien durante este click en el botón?
            self.actualizar_pantalla() # Repinta si hubo un cadáver.
            return # Fuerza corte estricto general del bloque.

        self.acciones_restantes -= 1 # Decrementa 1 del contador general (ej. 3 -> 2).
        if self.acciones_restantes <= 0: # Criterio Maestro: Si se te acabaron los turnos por andar disparando/comprando/curando...
            self.acciones_turno_anterior = self.clima.acciones_permitidas # Guardado estadístico al log de soporte.
            self.fase = "TURNO_IA" # Congela la base de la pantalla dictando que es hora del cerebro virtual.
            self.actualizar_controles() # Oculta la botonera humana y saca aviso bloqueador.
            self.log_ia = [] # Certeza extra de borrar el listado viejo narrativo.
            self.mensaje_sistema = "🤖 Analizando el radar... IA tomando el control." # Empieza su acto de teatro en el cajón superior de notificaciones.
            self.actualizar_pantalla() # Dispara visualmente la pantalla para meter presión.
            time.sleep(1.5) # Le quita fluidez e inyecta terror pausando tu juego 1.5 segs.

            porta_ia = [b for b in self.maquina.flota if b.tipo == 'portaaviones'][0] # Detecta variables vitales de barco enemigo.
            ingeniero_bot = [b for b in self.maquina.flota if b.tipo == 'ingeniero'][0] # Detecta vitales del médico IA.
            pesq_bot = [b for b in self.maquina.flota if b.tipo == 'pesquero'][0] # Detecta vitales de ganancia IA.
            columnas_letras = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"] # Array con decodificador texto para el final del ataque.

            if not pesq_bot.vivo: # CEREBRO IA PASIVA: Lo mismo que arriba para humanos. ¿Le mataste su mina de oro comercial?
                balas_ia = random.randint(1, 3) # Pues igual ella escarba sus cañones como buen IA.
                self.maquina.municion['cañon'] += balas_ia # Los roba para su inventario.
                self.log_ia.append(f"📦 El Pesquero IA hundido generó {balas_ia} Cañones extra.") # Narra el dolor en su registro.

            acciones_ia_totales = self.clima.acciones_permitidas # Solicita al objeto clima cuántos tiros le tocan en base al sorteo previo de inicio de ronda.

            for i in range(acciones_ia_totales): # Ciclo 'for' que itera por N turnos que tenga la IA por derecho del clima.
                if sum(1 for b in self.humano.flota if b.vivo) == 0: break # Break-safety: Si ya te fulminó al 1er tiro de 3, ignora los 2 restantes para no gastar balas ni hacerte Bullying en log.

                self.mensaje_sistema = f"⏳ La IA está ejecutando su jugada {i+1} de {acciones_ia_totales}..." # Cambia dinámicamente tu cuadro mientras hace el loop.
                self.actualizar_pantalla() # Tira esa orden arriba de inmediato.
                time.sleep(1.5) # Segundo y medio de terror por cada tiro de la máquina.

                balas_totales_ia = sum(self.maquina.municion.values()) # Extrae todos los cañones, artillerías y torpedos y los suma en int.
                criticos = [b for b in self.maquina.flota if b.vivo and b.vida_actual < b.vida_max] # Compila un objeto list con cada barco con daño de la Máquina.

                armas_permitidas_ia = ['cañon'] # Regla de Armas RPG. Todos arrojan cañones.
                if porta_ia.vivo: armas_permitidas_ia.extend(['artilleria', 'bombardeo']) # IA Inteligencia Artificial restringe su propia base pesada si le hundes el portaviones (Trampa de balance).
                if ingeniero_bot.vivo: armas_permitidas_ia.append('torpedo') # Si le hundiste al médico de la IA, ella no podrá lanzarte nunca un torpedo 60DMG.

                armas_disp_ia = [a for a in ['cañon', 'artilleria', 'bombardeo', 'torpedo'] if self.maquina.municion[a] > 0 and a in armas_permitidas_ia] # Compara la munición actual en su dict con la matriz de legalidad previa.

                if ingeniero_bot.vivo and criticos: # ÁRBOL DECISIONES IA (PRIORIDAD N1 - SOBREVIVIR): IA Evalúa la cura como prioritaria si tiene médico activo y heridas graves.
                    criticos.sort(key=lambda x: x.vida_actual) # Inteligencia Artificial escoge inteligentemente: Ordena de vida baja a alta para priorizar al moribundo.
                    poder_cura = 30 # Cura máxima establecida de 30 para ella también (Fair play).
                    criticos[0].vida_actual = min(criticos[0].vida_max, criticos[0].vida_actual + poder_cura) # Añade los 30 tapando hasta el tope legal.
                    self.restaurar_casilla(self.mapa_m, self.mapa_m_original, criticos[0].tipo) # Ejecuta borrar la marca roja para ella en la pantalla que TÚ ves (así percibes visualmente que te estropeó tu ataque).
                    self.log_ia.append(f"🛠️ La IA usó su Ingeniero para reparar su barco {criticos[0].nombre} (+{poder_cura} HP).") # Narrador lo cuenta en rojo oscuro en el log.

                elif len(armas_disp_ia) == 0 and self.maquina.creditos < 500 and pesq_bot.vivo: # CEREBRO IA (PRIORIDAD N2 - POBREZA CERO): Si no tengo balas Y no tengo 500 para la más barata Y tengo mi pesquero...
                    self.maquina.creditos += pesq_bot.creditos_generados # Uso todo el turno en farmear créditos como desesperada.
                    self.log_ia.append(f"💰 La IA activó su Pesquero y ganó +{pesq_bot.creditos_generados} CR.") # Mando al bot a notificar el farmeo para el humano rabioso.

                elif len(armas_disp_ia) == 0 and self.maquina.creditos >= 500: # CEREBRO IA (PRIORIDAD N3 - IR DE COMPRAS): Si estoy sin balas, pero ya junté platita...
                    arma_compra = 'cañon' # Inicializa compra cutre como comodín.
                    for a in reversed(['cañon', 'artilleria', 'bombardeo', 'torpedo']): # Lee todo del final (caro) a inicio (barato).
                        if a in armas_permitidas_ia and self.maquina.creditos >= self.info_armas[a]['costo']: # Compara su plata si le alcanza para el tope de su matriz de "armas legales".
                            arma_compra = a # Si tiene mil millones y está permitida compra la de 1500CR (Torpedo).
                            break # Corta el for para no degradarse a comprar cañones basuras.
                    self.maquina.creditos -= self.info_armas[arma_compra]['costo'] # Frena en caja a pagar restando al inventario IA.
                    self.maquina.municion[arma_compra] += 1 # Aumenta en 1 unidad la billetera de cañones de la IA.
                    self.log_ia.append(f"🛒 La IA compró 1x {arma_compra.capitalize()}.") # Lo sube al log para que tiembles (Oh no, me va a tirar un Torpedo).

                elif len(armas_disp_ia) > 0: # CEREBRO IA (PRIORIDAD N4 - CAZAR): Si por fin tiene balas, y nadie se está desangrando, y no anda pobre...
                    arma_usada = armas_disp_ia[-1] # Saca de la lista preordenada el arma de daño bestial de hasta el final (la más poderosa siempre).
                    self.maquina.municion[arma_usada] -= 1 # La extrae borrando 1 del inventario de dicts.
                    dmg = self.info_armas[arma_usada]['danio'] # Agarra el número numérico 30, 40, 50 o 60.

                    posibles_blancos = [] # PREPARA ALGORITMO CAZADOR-BUSCADOR.
                    for f in range(10): # Evalúa cada coordenada del mapa humano (Nivel Fila)
                        for c in range(10): # (Nivel Columna)
                            if self.mapa_h.matriz[f][c] == "X": # PREGUNTA CRÍTICA: ¿Ya hay una mancha de sangre (X) en este cuadrante del humano?
                                for df, dc in [(-1,0), (1,0), (0,-1), (0,1)]: # Si es SÍ, genera submatriz de adyacencias cruces perfectas (arriba, abajo, izquierda, derecha).
                                    nf, nc = f+df, c+dc # Crea la posible "Nueva Fila" y "Nueva Columna".
                                    if 0 <= nf < 10 and 0 <= nc < 10: # Evalúa si esa adyacencia no se estrella con la pared "11" imaginaria o pared "-1" para evitar bugs.
                                        if self.mapa_h.matriz[nf][nc] not in ["X", "O"]: # Criba final CAZADOR: ¿Esa zona pegada a mi sangre previa NO es ya otra sangre X, y tampoco un fallo de agua "O" que hice antes?
                                            posibles_blancos.append((nf, nc)) # LA RECOGE COMO UN BLANCO DE DESTRUCCIÓN. IA ATACARÁ AHÍ ASEGURANDO HUNDIMIENTO DEL BARCO O DETECCIÓN MULTIPLE.

                    if posibles_blancos: # PREGUNTA RESOLUTIVA: ¿Hubo adyacencias? (Básicamente, ¿La IA está rastreando un barco tuyo sangrante?).
                        f_r, c_r = random.choice(posibles_blancos) # Fuerza a la IA a arrojar el dardo SOLO hacia tus adyacencias ignorando 90 cuadros de agua.
                    else: # Si tu mapa está inmaculado, o sus intentos están vacíos, y el CAZADOR dio Lista Cero...
                        f_r, c_r = random.randint(0, 9), random.randint(0, 9) # Cierra los ojos y lanza a la buena de Dios un Random x Random.
                        intentos = 0 # Inicializador de prevención loop eterno.
                        while self.mapa_h.matriz[f_r][c_r] in ["X", "O"] and intentos < 20: # Mientras su dardo ciego aterrice en un fallo o X pasados (Evitando la IA pendeja).
                            f_r, c_r = random.randint(0, 9), random.randint(0, 9) # Fuerza resorteo incesante.
                            intentos += 1 # Escala variable protectora hasta 20 iteraciones (Si todo está lleno previene congelamiento y la ignora).

                    coord_texto = f"{columnas_letras[c_r]}{f_r + 1}" # Generador string B3 (Extrae letra array Letras, Saca Index Fila + 1 para que un índice 0 humano entienda '1').
                    celda = self.mapa_h.matriz[f_r][c_r] # Examina las tripas de la celda donde la IA definió su punto de impacto de hoy.

                    if celda not in ["X", "O"]: # Previene errores raros de iteración 20 por colapso (Casi imposible, protección).
                        if celda == "~": # ¿El impacto dio contra el agua cruda?
                            self.mapa_h.matriz[f_r][c_r] = "O" # Si sí, graba la redonda O (Agua) para que tú la veas y rías.
                            self.log_ia.append(f"💦 La IA disparó un {arma_usada.capitalize()} a [{coord_texto}] y falló.") # Carga a su panel que erró.
                        else: # ¿Le dio al pesquero o portaaviones (letras variadas)?
                            self.mapa_h.matriz[f_r][c_r] = "X" # Saca la pluma y marca "X" de sangre.
                            mapa_b = {"P":"portaaviones", "E":"ingeniero", "S":"pesquero", "I":"inteligencia"} # Genera mini-diccionario decodificador de letrucha a ClaseBarco.
                            obj = [b for b in self.humano.flota if b.tipo == mapa_b[celda]][0] # Lee la 'letrucha', la saca a TipoBarco de la Flota Humana y extrae la iteración list cruda [0] objeto (La propia clase de BarcoPython).
                            obj.recibir_danio(dmg) # Usa las tripas de la FASE 1 para llamar su función de recibir daño puro con el parámetro 'dmg' según si usó torpedo o artillería IA.
                            self.log_ia.append(f"💥 ¡CUIDADO! La IA acertó un {arma_usada.capitalize()} en tu {obj.nombre} [{coord_texto}] ({dmg} DMG).") # Da el chivatazo trágico por texto.
                    else: # Excepción (Colapso): IA pegó a sus ruinas viejas (solo posible tras iteración 20).
                        self.log_ia.append(f"📡 La IA gastó un {arma_usada.capitalize()} atacando ruinas en [{coord_texto}].") # Informa fallo estúpido de máquina saturada.
                else: # PREVENCIÓN (Si no sirvió ni para curar, ni platita, ni balas). Error global máquina.
                    self.log_ia.append(f"⚠️ La IA se ha quedado sin opciones. Pierde su acción.") # Narra un salteo de turno puro y duro.

                self.actualizar_pantalla() # Ordena un nuevo marco render tras ESTA acción específica que tomó.
                time.sleep(1) # Le da al usuario humano 1 mísero segundo de lectura del desastre que causó el bombazo para que sude adrenalina.

            if self.verificar_victoria(): # Acabaron sus turnos. Llama al Juez Supremo del Juego.
                self.actualizar_pantalla() # Si el juez devolvió True (IA te hundió TODO el último milisegundo de su turno múltiple), actualiza muerte humana.
                return # Destruye y acaba proceso finalizando el juego de golpe sin ejecutar el resto.

            self.mensaje_sistema = "✅ Turno enemigo finalizado. Analiza el reporte de daños." # Tras su tanda, notifica al humano tranquilizante.
            self.fase = "PAUSA_RONDA" # Dispara nueva Fase Visual de "Caja Input para Pausa con Enter".
            self.actualizar_controles() # Renderiza los widgets obligando bloqueo y solo caja pausa enter.
            self.actualizar_pantalla() # Dibuja lo anterior dictado.

        else: # FLUJO DE CAJA: Si la variable de descuento bajó el tiro de 3 a 2 acciones... (sigue siendo el turno del humano).
            self.actualizar_pantalla() # Únicamente repintaría para que sigas oprimiendo botones sin interrupción de la IA (Turno normal Humano en loop de botonazos).

    def procesar_ataque_h(self, b): # EVENTO MAESTRO: Clic en botón "Atacar" tuyo.
        if self.clima.ataques_bloqueados: # Condición "Paz Mundial", "Cese Fuego", Clima anulador de Fase 4?
            self.mensaje_sistema = "🕊️ Bloqueado por evento: Cese al Fuego activo." # Arroja la variable.
            self.actualizar_pantalla(); return # Reanuda sin tocar tú acción.

        indices = traducir_coordenada(self.txt_coord.value) # Manda al motor la caja de tu imput a que extraiga coordenadas en números.
        if indices is None: # Si te equivocaste...
            self.mensaje_sistema = "❌ Coordenada inválida. Escribe una letra de la A-J y un número del 1-10 (Ej: E5)." # Sistema anti-error total de consola humana inmersa.
            self.actualizar_pantalla(); return # Reanuda.

        porta_vivo = [b for b in self.humano.flota if b.tipo == 'portaaviones'][0].vivo # Regla RPG 1 de Bloqueo Armas pesadas: Obtiene valor Vida Portaaviones.
        ing_vivo = [b for b in self.humano.flota if b.tipo == 'ingeniero'][0].vivo # Regla RPG 2 Torpedos: Obtiene valor vivo de tu Ingeniero de mecánica pesada.

        armas_permitidas = ['cañon'] # Establece array base del comodín que nunca muere.
        if porta_vivo: armas_permitidas.extend(['artilleria', 'bombardeo']) # Suma con 'extend' los dos string de armas monstruosas si el buque bandera portaaviones respira.
        if ing_vivo: armas_permitidas.append('torpedo') # Agrega torpedo si ingeniero vive.

        armas_disponibles = [a for a in ['cañon', 'artilleria', 'bombardeo', 'torpedo'] if self.humano.municion[a] > 0 and a in armas_permitidas] # List Comprehension que revisa dos condiciones: Si posees la munición comprada Y TAMBIEN es un arma legal permitida de lanzar sin restricciones de bloqueos.

        if not armas_disponibles: # Si la lista está vacía...
            if sum(self.humano.municion.values()) > 0: # Pero si tienes balas globales reales en la matriz diccionario... (Significa que tu arsenal pesado comprado es inútil por tus barcos muertos).
                self.mensaje_sistema = "❌ Munición bloqueada. Tus barcos pesados están hundidos. Usa cañones." # Lanza castigo verbal por tu falta de habilidad táctica.
            else: # O si la culpa es tuya porque se te olvidó ir a tienda...
                self.mensaje_sistema = "❌ ¡No tienes munición disponible! Ve a la tienda." # Te informa de la tontería.
            self.actualizar_pantalla(); return # Cierra función para que repienses.

        arma_usada = armas_disponibles[-1] # Elije de las balas tu artillería más salvaje disponible al final de la lista.
        self.humano.municion[arma_usada] -= 1 # La retira formalmente del registro.
        dmg = self.info_armas[arma_usada]['danio'] # Agarra los 30-40-50-60 valores para impactar luego en la función ajena.

        fila, col = indices # Desempaqueta las coordenadas tuyas (Ej: C2).
        celda = self.mapa_m.matriz[fila][col] # Verifica a qué de las miles de celdas de la máquina le estás intentando acertar.
        if celda in ["X", "O"]: # Si el humano tarado atinó a donde él mismo ya había disparado antes (X de sangre o O de agua)...
            self.mensaje_sistema = "⚠️ Error: Ya habías bombardeado esa coordenada previamente." # Salva al humano de quemar una bala real.
            self.actualizar_pantalla(); return # Detiene ciclo y perdona la vida.

        if celda == "~": # ¿Tu bombazo atinó al puro mar?
            self.mapa_m.matriz[fila][col] = "O" # Cambia la tilde ~ por la 'O' de fallo visual.
            self.mensaje_sistema = f"💦 Fallo. Disparaste un {arma_usada.capitalize()} al mar abierto." # Notifica chapoteo del arma.
        else: # ¡Atinaste!
            self.mapa_m.matriz[fila][col] = "X" # Reescribe la vieja P/E/S/I escondida a una "X" gigante para tu radar y el de ella.
            mapa_b = {"P":"portaaviones", "E":"ingeniero", "S":"pesquero", "I":"inteligencia"} # Busca qué letra decodifica con tu impacto ciego a través de la niebla de guerra.
            victima = [barco for barco in self.maquina.flota if barco.tipo == mapa_b[celda]][0] # Cruza esa Letra de víctima y la atrapa del registro en Memoria de la máquina enemiga.
            victima.recibir_danio(dmg) # LLama a la función del enemigo para destrozarle internamente la cifra de los Stats "Vida" en base a la variable extraída 'dmg' (ej: 60 del torpedo).
            self.mensaje_sistema = f"💥 ¡IMPACTO! Acertaste con un {arma_usada.capitalize()} en un barco enemigo ({dmg} DMG)." # Celebra con rojo intenso en el panel comandante el acierto con el daño hecho visualizado.

        self.txt_coord.value = "" # Vuelve tu casillita blanca de escribir al estado original.
        self.descontar_accion() # ¡El tiro costó una acción! Llama a la cajera maestra "descontar_accion" a ver si sigues vivo y tirando o si entra la máquina vengadora.

    def procesar_sigilo_h(self, b): # EVENTO DEL BOTÓN AZUL RADAR MÁGICO.
        intel = [barco for barco in self.humano.flota if barco.tipo == 'inteligencia'][0] # Evalúa barco de inteligencia (Submarino Sigiloso o Bote de Radares).
        if not intel.vivo: # REGLA DE JUEGO RPG: Hundido barco especial, anulada magia.
            self.mensaje_sistema = "❌ Tu Inteligencia está hundida. Radar desactivado." # Falla el uso del radar para el llorón de turno.
            self.actualizar_pantalla(); return # Reanuda.

        enemigos_vivos = [barco for barco in self.maquina.flota if barco.vivo] # Generador para filtrar barcos para detectar el "verdadero" y "falso" radar.
        if not enemigos_vivos: return # Anti-crash: Para un milagro en el que se hundió la IA pero diste click aquí a 1000 clics por segundo.

        objetivo = random.choice(enemigos_vivos) # Tira al azar un barco base a exponer (ej: pesquero).
        coords_reales = [] # Base espía real.
        letras = {"portaaviones":"P", "ingeniero":"E", "pesquero":"S", "inteligencia":"I"} # Conversor.
        letra = letras[objetivo.tipo] # Extrae la P/S/E.

        for f in range(10): # Mapea todo el juego en milisegundos buscando la maldita letra:
            for c in range(10):
                if self.mapa_m_original[f][c] == letra and self.mapa_m.matriz[f][c] != "X": # Dos condiciones: ESTÁ la letra allí en el radar de inteligencia virgen (foto tomada en colocación), Y también comprueba si tu radar de guerra ya no la dañó a una 'X' previamente (que sería pendejo avisarte de un lugar roto obvio).
                    coords_reales.append((f, c)) # La detectó, te la añade a radar.

        if not coords_reales: coords_reales.append((0,0)) # Excepción contra Matrix Out of Bounds en radares vacíos.
        coord_real = random.choice(coords_reales) # Escoge a random uno solo de todos los trozos del barco para revelar. (Si no, te daba el portaaviones de lado a lado completo regalado).

        coords_falsas = [] # Crea basurero fake (pings falsos de radar).
        while len(coords_falsas) < 4: # Se queda aquí atrapado generando 4 pings incesantemente.
            fr, cr = random.randint(0, 9), random.randint(0, 9) # Fabrica el engaño.
            if (fr, cr) != coord_real and (fr, cr) not in coords_falsas: # Verifica: ¿Este dardo random dio por error a mi coordenada verdadera real? Y... ¿Acaso ya había repetido esta misma mentira? (Buscamos 4 mentiras distintas al final del día).
                coords_falsas.append((fr, cr)) # Añade las fakes filtradas exitosamente.

        todas = [coord_real] + coords_falsas # Construye matriz combinada con 1 verdad, 4 engaños.
        random.shuffle(todas) # Algoritmo coctelera para que jamás sepas si la de en medio de la caja azul del GUI es la real o la primera o la última.

        columnas_letras = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"] # Lógica decodificadora para imprimir "A5".
        textos = [f"{columnas_letras[c]}{f+1}" for f, c in todas] # Función Python para pasar el vector bruto (0,1) al visual C2 con bucle 'for' incorporado adentro de listas.

        self.mensaje_radar = f"Señales detectadas en {', '.join(textos)}. (Una es real)." # Inyecta a la variable global permanente el string que genera el bloque azulado persistente para la ronda.
        self.radar_turnos_vida = 2 # El bloque vivirá por 2 cambios climáticos (Toda la ronda de la máquina y tuyos subsiguientes).
        self.mensaje_sistema = "🕵️‍♂️ Radar Sigiloso activado. Cediendo tus acciones a la IA..." # Informa.

        self.acciones_restantes = 0 # El precio más caro: Te deja tu barra de acciones de 3 o 2 a cero patatero de inmediato. (Es un todo o nada en radar).
        self.descontar_accion() # Invoca al demonio del cajero maestro para que te eche al instante porque te gastaste todo en "0".

    def procesar_reparacion_h(self, b): # EVENTO DEL BOTÓN VERDE REPARAR CINTA ADHESIVA.
        ing = [barco for barco in self.humano.flota if barco.tipo == 'ingeniero'][0] # Agarra tu soldador y mira a ver si está flotando o chapoteando en el fondo.
        if not ing.vivo: # Si es lo último...
            self.mensaje_sistema = "❌ Tu Ingeniero está hundido. No puedes reparar tu flota." # Rebota pidiendo perdon y niega la reparación al estúpido almirante.
            self.actualizar_pantalla(); return # Reanuda.

        criticos = [barco for barco in self.humano.flota if barco.vivo and barco.vida_actual < barco.vida_max] # Genera array lista que revisa la flota completa filtrada buscando heridos y vivos (no levanta de los muertos pero sana).
        if not criticos: # Si todos estaban con 100/100...
            self.mensaje_sistema = "⚠️ Toda tu flota ya tiene la vida al máximo. No hay nada que reparar." # Freno del anti error general, te salva de gastar turno a lo idiota.
            self.actualizar_pantalla(); return # Sale de la función y no resta acción de clima (vuelves a elegir un botón sin gastar penalización).

        criticos.sort(key=lambda x: x.vida_actual) # Algoritmo Ordenador Priorizado por Criterio Key = lambda que dice en español: Ordename de Vida Chiquita a Grande.
        objetivo = criticos[0] # Agarra el número cero de esa lista (es decir, el barco que se estaba muriendo desangrado, a punto de fallecer y bloquearte el Portaaviones).

        poder_cura = 30 # Cura maestra balanceada (fija como regla).
        objetivo.vida_actual = min(objetivo.vida_max, objetivo.vida_actual + poder_cura) # Recupera HP interno del barco (pero limitado usando el min() para que nunca se cure "hasta 180" si su max original era "100").
        self.restaurar_casilla(self.mapa_h, self.mapa_h_original, objetivo.tipo) # Tira llamada maestra que invoca la función Restaurar_casilla pasándole el mapa destrozado, la fotocopia guardada virgen inmaculada de tus 4 barquitos, y la variable del barco sanado actual para que vaya y devuelva el daño rojo.

        self.mensaje_sistema = f"🛠️ Riesgo asumido: Reparaste tu {objetivo.nombre.capitalize()} (+{poder_cura} HP). Una casilla fue ocultada." # Celebra el reparo y avisa riesgo general (te perdiste lanzar bombas).
        self.descontar_accion() # Invoca el cobro contable de acción (-1 tiro de los permitidos).

    def procesar_pesquero_h(self, b): # BOTON CLARO DE COMERCIAR Y GANAR PLATITA.
        pesq = [barco for barco in self.humano.flota if barco.tipo == 'pesquero'][0] # Mira al pesquero y define estado en variable rápida.
        if pesq.vivo: # ¿Vivo?
            self.humano.creditos += pesq.creditos_generados # Toma de la variable de RPG sus créditos ganados (o 500, o de pronto 750 si ya era Nivel 3) y los inyecta en la vena Humana de self.
            self.mensaje_sistema = f"💰 Recaudación exitosa. Sumaste +{pesq.creditos_generados} CR a tus fondos." # Celebra los billullos informados por texto visual.
            self.descontar_accion() # Cobra acción. (Gastaste tiempo farmeando).
        else: # Ahogado y trataste de farmear?
            self.mensaje_sistema = "❌ Error: Tu barco pesquero está hundido." # Error para tontos. Te respeta la jugada por si fue Miss-Click.
            self.actualizar_pantalla() # Repinta a lo normal.

    def procesar_compra_h(self, b): # BOTÓN COMPRAR "SHOPPING-CART".
        arma = self.drop_tienda.value # Pide el valor guardado en el DropDownMenu list seleccionado (Ej: Si seleccionó 'Bombardeo (900)', te arroja la variable oculta 'bombardeo').
        porta_vivo = [b for b in self.humano.flota if b.tipo == 'portaaviones'][0].vivo # Regla RPG 1 Tienda.
        ing_vivo = [b for b in self.humano.flota if b.tipo == 'ingeniero'][0].vivo # Regla RPG 2 Tienda.

        if arma in ['artilleria', 'bombardeo'] and not porta_vivo: # Pregunta de oro: Está pidiendo armas para hundidos?
            self.mensaje_sistema = f"❌ Tu Portaaviones está hundido. No puedes comprar armas pesadas." # Freno por buque perdido.
            self.actualizar_pantalla(); return # Rechaza cobro acción y dinero.
        if arma == 'torpedo' and not ing_vivo: # Pregunta si piden torpedos pero sin su ingeniero a la mano:
            self.mensaje_sistema = f"❌ Tu Ingeniero está hundido. No puedes comprar Torpedos." # Frena compra.
            self.actualizar_pantalla(); return # Rebota la acción respetando todo y pidiendo que vuelvan a usar un botón con la plata.

        costo = self.info_armas[arma]['costo'] # Busca con llave primaria de arma ("torpedo") para jalar solo el valor secundario de ['costo'] asociado, (Ej: 1500).
        if self.humano.creditos >= costo: # Verifica con el tendero de caja si tienes saldo:
            self.humano.creditos -= costo # Si sí, disminuye saldo Humano.
            self.humano.municion[arma] += 1 # Aumenta 1 arma literal para el inventario de la llave (Ej: Torpedos)
            self.mensaje_sistema = f"✅ Compraste 1x {arma.capitalize()} por {costo} CR. (Fondos y munición actualizados)" # Registra a modo contable finalizada la compra en texto GUI.
            self.descontar_accion() # Cobra tu movimiento de ir a la tienda con tiempo.
        else: # Si vas al tendero pobre de mendigo...
            self.mensaje_sistema = f"⛔ Fondos insuficientes para comprar {arma.capitalize()} ({costo} CR)." # Da aviso que no tienes CR, y por eso el fondo es rojo prohibido.
            self.actualizar_pantalla() # Re-carga todo y no gastas turno al menos en un missclick ciego.

    def procesar_pasar_turno(self, b): # BOTON "SALTAR TURNO". (Rendición pacífica temporal o acopio táctico de espera porque tienes radar).
        self.mensaje_sistema = "⏩ Cediendo acciones restantes al enemigo..." # Cambia la leyenda del log y suelta lo inevitable.
        self.acciones_restantes = 0 # Bota tus valiosas acciones a la basura sin remordimiento para detonar IA con un valor cero 0 en el loop central de Cajero.
        self.descontar_accion() # Envía a procesar la condena y salta directo a TURNO IA.

# ============================================================================ # Comentario base final final visual estético de remate.
# DISPARADOR AUTOMÁTICO DE LA INTERFAZ # Define a humanos lo que ocurre post-clase maestro para iniciar todo de una vez.
# ============================================================================ # Final del cuadro separador.

clear_output() # Función para pre-eliminar suciedad remanente de ejecuciones de celdas antiguas muertas en el historial del Colab.
mapa_h_web = TableroReal(jugador_humano.nombre) # Invoca código ajeno de fases anteriores para generar la clase real de matriz "H".
mapa_m_web = TableroReal(jugador_maquina.nombre) # Invoca código de fases previas para "M".

app_naval = MotorJuegoWidgets(jugador_humano, jugador_maquina, mapa_h_web, mapa_m_web, motor_clima) # ¡El nacimiento definitivo! Instancia el Motor (Crea todo el monstruo con sus cientos de variables). Pone todo este Frankenstein vivo dentro de `app_naval`.

display(app_naval.contenedor_principal) # Llama a la herramienta mágica widget Notebook de dibujar el corazón total `contenedor_principal` en la hoja en limpio para que arranque Fase 1 Despliegue de Barcos o todo termine de renderizar.

# ================= FIN DEL CÓDIGO ================= # Acaba y te marca límite exacto donde termina toda la celda Python Colab.